### Config

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from hydra import compose, initialize
from omegaconf import OmegaConf

initialize(config_path="../config", version_base="1.3")
cfg = compose(config_name="config")
print(OmegaConf.to_yaml(cfg))

sys.path.insert(0, str(cfg.paths.project_root))

paths:
  project_root: /home/p84400019/projects/consciousness-llms/IT-LLMs/
  model_path: ${model.company}/${model.model_family}/${model.model_size}/${model.it}/
  activation_method: ${time_series.node_type}/${time_series.node_activation}/${time_series.projection_method}/
  phyid_method: ${paths.activation_method}phyid_tau-${phyid.tau}/phyid_kind-${phyid.kind}/phyid_redundancy-${phyid.redundancy}/
  deactivation_method: deactivate_k_nodes_per_iteration-${deactivation_analysis.deactivate_k_nodes_per_iteration}/max_deactivated_nodes-${deactivation_analysis.max_deactivated_nodes}/
  data_dir: ${paths.project_root}data/${paths.model_path}${generation.name}/
  data_activations_dir: ${paths.data_dir}activations/
  data_activations_file: ${paths.data_activations_dir}multi_prompt_activations.pkl
  data_phyid_dir: ${paths.data_dir}phyid/${paths.phyid_method}
  data_phyid_file: ${paths.data_phyid_dir}multi_prompt_phyid.pkl
  data_phyid_file_data_array: ${paths.data_phyid_dir}multi_prompt_phyid.n

### Loading the model

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, AutoConfig
from src.utils import perturb_model, randomize_model_weights

load_model = True

model_name = cfg.model.hf_name
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    trust_remote_code=True
)
if load_model:
    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        device_map='auto', 
        attn_implementation='eager',  
        trust_remote_code=True,
        revision=cfg.model.revision if hasattr(cfg.model, 'revision') else None,
    )
    if cfg.model.it == 'random':
        # randomize_model_weights(model, mean=0.0, std=0.02)
        perturb_model(model, scale=10.0)
    # model.generation_config = GenerationConfig.from_pretrained(model_name)
    # model.generation_config.pad_token_id = model.generation_config.eos_token_id
    model.eval()
print(type(model))
print(model)
print(model.config)


/home/p84400019/miniconda3/envs/int/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.88s/it]


<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
   

### Record activations, save them, and verify them

In [3]:
from src.activation_recorder import ActivationRecorder, MultiPromptActivations

load_from_disk = False
data_activations_file = cfg.paths.data_activations_file
max_new_tokens=cfg.generation.max_new_tokens
prompts = cfg.generation.prompts
# if it not a list but a list of lists, flatten it
if isinstance(prompts, list) and all(isinstance(p, list) for p in prompts):
    prompts = [item for sublist in prompts for item in sublist]

prompt_template = cfg.model.apply_chat_template
print(f"Using prompt_template: {prompt_template}")

if not load_from_disk:
    recorder = ActivationRecorder(model, tokenizer)
    activations = recorder.record_prompts(prompts, max_new_tokens=max_new_tokens, prompt_template=prompt_template)
    print(f"Recorded {len(activations)} activations for {len(prompts)} prompts.")
    activations.save(data_activations_file)
    activations.verify_recorded_activations(prompts=prompts, max_new_tokens=max_new_tokens, tokenizer=tokenizer, diff_q_size=True)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Using prompt_template: base
Recorder initialized for model: ModelInformation(model_name='meta-llama/Llama-3.1-8B', model_architecture='LlamaForCausalLM', num_layers=32, num_attention_heads_per_layer=32, total_num_attention_heads=1024, hidden_size=4096, head_dim=128, n_routed_experts=None, n_shared_experts=None, num_experts_per_tok=None, attention_implementation='default')
Working on prompt 0: Question: If you have 15 apples and you give away 5, how many do you have left?
 Answer: 



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt 0 completed: Question: If you have 15 apples and you give away 5, how many do you have left?
 Answer: 10 apples
Explanation: Subtraction is one of the four basic mathematical operations. The others are addition, multiplication and division. Subtraction is the reverse of addition. Subtraction finds the difference, or the change in value, between two numbers.
The most common way to represent subtraction in writing is to place a minus sign (−) between the two numbers. For example, 5 − 3 = 2. This implies that the result is 2, because 5 minus 3 is equal to 2. Here, 5 is called the subtrahend and 3 is called the minuend.
Subtraction

Working on prompt 1: Question: A rectangle's length is twice its width. If the rectangle's perimeter is 36 meters, what are its length and width?
 Answer: 



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Prompt 56 completed: Question: Describe a character who is trying to make amends for past actions.
 Answer: 1. He has a good heart but he has done something very bad. 2. He is trying to make amends. 3. He is trying to atone for his sins. 4. He is trying to make up for his mistakes. 5. He is trying to redeem himself. 6. He is trying to prove himself. 7. He is trying to show that he is not a bad person. 8. He is trying to show that he has changed. 9. He is trying to show that he is worthy of forgiveness. 10. He is trying to show that he is worthy

Working on prompt 57: Question: Write about a character who overcomes a fear with the help of a friend.
 Answer: 



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt 57 completed: Question: Write about a character who overcomes a fear with the help of a friend.
 Answer: 1. I have a friend named Mary. She is very much afraid of dogs. She does not like to go out with me because I have a pet dog. She always tells me to leave my dog at home when we go out. She thinks that my dog will bite her. She has this fear since she was a child. She has a very good friend named Susan. She tells her about her fear and asks her to accompany her when she goes out. She tells her that she will not allow her to go out if she does not accompany her. Mary tells her that she will not be able to control herself if she sees a

Working on prompt 58: Question: Create a story about a misunderstanding between characters from different cultures.
 Answer: 



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt 58 completed: Question: Create a story about a misunderstanding between characters from different cultures.
 Answer: 1. The story begins with two characters from different cultures meeting for the first time. They are both excited to learn about each other's customs and traditions.
2. However, during the conversation, they start to misunderstand each other's words and actions. For example, one might think that the other is being rude when they are simply being polite in their own culture.
3. As the misunderstanding continues, it begins to cause tension between the two characters. They start to argue and get frustrated with each other.
4. Eventually, they realize that they have been misinterpreting each other's words and actions due to their cultural differences. They apologize

Working on prompt 59: Question: Imagine a scenario where a character has to forgive someone who wronged them.
 Answer: 

Prompt 59 completed: Question: Imagine a scenario where a character has to forgive 

In [ ]:
# Load the activations from disk.
loaded_activations = MultiPromptActivations.load(data_activations_file)

# Optional: verify the loaded activations match the saved ones.
print("Loaded MultiPromptActivations object has:", len(loaded_activations.prompts), "prompts recorded.")

# Check again the activations
loaded_activations.verify_recorded_activations(prompts=prompts, max_new_tokens=max_new_tokens, tokenizer=tokenizer, diff_q_size=True)

print("Final MultiPromptActivations object has:", len(loaded_activations.prompts), "prompts recorded.")

# Extract the first prompt, first step, first layer, first head
prompt_acts = loaded_activations.prompts[0]
step_acts = prompt_acts.steps[0]
layer_acts = step_acts.layers[0]
attn = layer_acts.attention
for head_idx, head_acts in attn.heads.items():
    print(f"Head {head_idx} activations:")
    print(head_acts.query.shape)
    print(head_acts.attention_weights.shape)
    print(head_acts.attention_outputs.shape)
    print(head_acts.projected_outputs.shape)

In [ ]:
step_acts = prompt_acts.steps[4]
for layer_idx, layer_acts in step_acts.layers.items():
    print(f"Layer {layer_idx} activations:")
    moe_layer_acts = step_acts.layers[2].moe
    for expert_id, expert_acts in moe_layer_acts.experts.items():
        print(f"Expert {expert_id} activations:")
        print(repr(expert_acts))
        print("Gate value:", expert_acts.gate_value)
        print("MLP output:", expert_acts.mlp_output)
        print("Expert output:", expert_acts.expert_output)

Layer 0 activations:


AttributeError: 'NoneType' object has no attribute 'experts'